# 410 — Zone discovery (blob overlays → manual windows)

The first, **exploratory** step of the confirmatory pooling pipeline. It loads the **same
canonical samples the clustering pipeline uses** (one sample = the trial-averaged ERSP per
electrode × condition, shape 129 × 300), overlays every contact's segmented blobs so you can
*see* where activity concentrates in time, and ends with a cell where **you hand-define** the
pooling windows.

**The warped time axis.** Each ERSP is time-warped **50% stimulus / 50% response**, so the
vertical dashed line at bin **150 (50%)** marks **response onset**: left of it = sensing /
perception, right of it = response / production.

**What to look for.** Each blob is drawn as a **red (positive) / blue (negative) ellipse
outline**, its shade scaled by the blob's mean dB. Stacked across all contacts, dense bands of
outlines reveal the time regions where the dataset reliably responds — the candidate **pooling
zones**. The time-marginal plot underneath quantifies the same thing (contacts active per bin).

**The goal:** read these, then in the final cell type the boxcar + Gaussian windows for the
three zones — **perception**, **pre_articulation**, **audio** — saved to
`outputs/pooling/window_config.json` for `420` to pool over.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


## 1 — Load the canonical dataset (identical to clustering)
Ungated: every electrode × condition ERSP for patients with all three conditions. Windowed
gating happens later in `420`, so we keep everything here. The heavy walk is cached.


In [ ]:
df_meta, X_3d = P.prepare_pooling_dataset(INPUT_DIR)
print('samples:', len(df_meta), '| X_3d:', X_3d.shape)
print('patients:', sorted(df_meta.patient_id.unique()))
df_meta.head()


## 2 — Blob overlays per condition
Red = positive (activation), blue = negative (suppression); outline shade ∝ |mean dB|. Dense
vertical bands = candidate pooling zones. ⚠️ Blobs are arbitrary shapes — the ellipse is a
moment-matched approximation (centre = centroid, axes = 2·std), so treat it as a *summary*
of each blob's extent, not its exact outline. (Segmenting every contact is the slow step.)


In [ ]:
disc = P.new_run_dir('discovery')
print('discovery run:', disc)
for cond in P.CONDITIONS:
    png = P.plot_blob_overlay(X_3d, df_meta, cond, disc / f'overlay_{cond}.png')
    display(Image(filename=str(png)))


## 3 — Time-marginal blob density
The quantitative read-off: how many contacts have a positive (up, red) or negative (down, blue)
blob active at each time bin. Peaks/plateaus here are exactly the windows worth pooling.


In [ ]:
for cond in P.CONDITIONS:
    png = P.plot_time_marginal(X_3d, df_meta, cond, disc / f'time_marginal_{cond}.png')
    display(Image(filename=str(png)))


## 4 — Define the pooling windows  ✍️  (edit this cell)
Fill in the three zones from what you saw above. **All numbers are percentages of the 0–300
warped axis** (50% = response onset). Each zone needs **both** a `boxcar` (the primary, equal-
weight window — `t_lo_pct`/`t_hi_pct`) **and** a `gaussian` (the robustness window —
`center_pct`/`sigma_pct`). The seed values below are placeholders; overwrite them.

> The pooling is *permissive by design*: each zone targets activity that is **definitely
> present** in its window — it does not try to be exclusive about activity elsewhere.


In [ ]:
cfg = {
    "schema_version": 1,
    "n_time": P.N_TIME,
    "stim_frac": P.STIM_FRAC,           # 50% mark = response onset = bin 150
    "axis_units": "percent_of_300_warped_axis",
    "zones": {
        # zone               boxcar (equal weight)         gaussian (centre-weighted)
        "perception":       {"boxcar": {"t_lo_pct": 0,  "t_hi_pct": 50},
                             "gaussian": {"center_pct": 25, "sigma_pct": 10}},
        "pre_articulation": {"boxcar": {"t_lo_pct": 40, "t_hi_pct": 60},
                             "gaussian": {"center_pct": 50, "sigma_pct": 8}},
        "audio":            {"boxcar": {"t_lo_pct": 50, "t_hi_pct": 100},
                             "gaussian": {"center_pct": 75, "sigma_pct": 12}},
    },
}
P.validate_window_config(cfg)
cfgp = P.save_window_config(cfg, P.OUTPUTS_ROOT / 'window_config.json')
print('saved ->', cfgp)
P.plot_window_preview(cfg);
